# QF 627 Programming and Computational Finance
## Lesson 03 | A Gentle Introduction to Quantitative Trading Strategies and Backtesting

> Hi Team 👋

> Thank you for opening the script 🙂 As we discussed, to maximize your learning, instead of covering each week’s lecture note in just one week, we will now be discussing each script over the course of two weeks.

> Today, we’ll begin exploring your first algorithmic trading strategy. Next week, we’ll continue building on that foundation. In the meantime, I encourage you to revisit the content and experiment with the code included in the script as a preview of the learning to come.

## DEPENDENCIES

In [ ]:
# Load libraries.

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

import seaborn as sns

import time

import datetime as dt
import re

import warnings
warnings.filterwarnings("ignore")

# Setting baseline seed
np.random.seed(2025)

# Set print options.

np.set_printoptions(precision = 3)

plt.style.use("ggplot")

mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.alpha"] = 0.25

mpl.rcParams["axes.facecolor"] = "white"

mpl.rcParams["legend.fontsize"] = 14

%matplotlib inline

# Define our customized timer function

def countdown(Time):
    
    while Time:
        minutes, seconds = divmod(Time, 60)
        timer = "{:02d}:{:02d}".format(minutes, seconds)
        
        print(timer, end = "\r")
        time.sleep(1)
        Time -= 1
        
    print("Let us solve the problem above together :)")

In [ ]:
%whos

## 👉 <a id = "top">Learning Pointers</a> 👈 

## [1. Rolling Statistics Revisited](#p1)

> ### <font color = red> Rolling Statistics </font>

## [2. Building Simple Momentum Trading Strategy](#p2)

> ### <font color = red> SMA </font>

## [3. Backtesting Trading Strategy](#p3)

> ### <font color = red> Backtesting is NOT Forecasting </font>

## [4. Performance Metrics](#p4)

> ### <font color = red> Sharpe, MDD, & CAGR </font>

## [5. What We Learned](#p5)

> ### <font color = red> Thus far... </font>

## <a id = "p1">1. </a> <font color = "green"> Rolling Statistics </font>  [back to table of contents](#top)

### Install new library for extracting financial market data from Yahoo Finance

In [ ]:
# !pip install yfinance==0.2.58

<mark>Workflow</mark>

- Define
- `Collect`

In [ ]:
%whos

In [ ]:
import yfinance as yf

In [ ]:
%who

In [ ]:
yf.__version__

### COLLECT & IMPORT data from Yahoo!Finance

In [ ]:
starting = "2025-01-01"
ending = "2025-05-06"

In [ ]:
nvda =\
( # one ticker
    yf
    .download("NVDA", # ticker
              start = "2025-01-01", # starting date
              end = "2025-05-06") # ending date
    # .droplevel("Ticker",
    #            axis = 1)
    # [["Close", "Volume"]]
)

In [ ]:
nvda

In [ ]:
(
    nvda
    .droplevel("Ticker",
               axis = 1)
)

In [ ]:
nvda.columns

In [ ]:
nvda[["Close"]]

In [ ]:
nvda["Close"]

In [ ]:
MAANG =\
    ["META",
     "AAPL",
     "AMZN",
     "NFLX",
     "GOOG"]

In [ ]:
maang_prices =\
(
    yf
    .download(MAANG,
              start = starting,
              end = ending)
    ["Close"]
)

In [ ]:
maang_prices

In [ ]:
## Extracting date for two symbols
two_symbols_meta_msft =\
(
    yf
    .download(["META", "MSFT"],
              start = starting,
              end = ending 
             )
    ["Close"]
)

In [ ]:
two_symbols_meta_msft.columns

In [ ]:
two_symbols_meta_msft

### Analytics Contexts


> `Rolling statistics` are useful in the `technical analysis` of stocks. Their use can be compared to `fundamental analysis`, which focuses on financial reports and the strategic positions of the company whose stock is being examined.

> The use of `two simple moving averages` (SMAs) is a basic trading strategy based on technical analysis.

•	A trader should go `long` on a stock (or any financial instrument in general) when its `shorter-term SMA is above its longer-term SMA`, and should go short when the opposite holds true.

•	This concept can easily be implemented with `pandas`, thanks to the `DataFrame object`.

> Rolling statistics are generally used only when there is enough data, given the window parameter specification.

#### Let's calculate the values for the shorter-term and longer-term SMAs.

> Then visualize the stock price data, along with the two SMA time series.

In [ ]:
%whos

In [ ]:
maang_prices

```python
# Moving Average Calculation
DF["ma_NofDays"] =\
(
    DF
    ["price"]
    .rolling(window = 22) # rolling 22 days --> 22 trading days ==> about a month
    .mean() # rolling averages with 22 days of windows
)

In [ ]:
apple_prices =\
(
    maang_prices
    [["AAPL"]]
)

In [ ]:
apple_prices

In [ ]:
apple_prices["ma_5"] =\
(
    apple_prices
    ["AAPL"]
    .rolling(window = 5) # 5 day MA --> one week of trading window
    .mean()
)

In [ ]:
apple_prices.head(15)

In [ ]:
apple_prices["ma_22"] =\
(
    apple_prices
    ["AAPL"]
    .rolling(window = 22) # 22 day MA --> one MONTH of trading window
    .mean()
)

In [ ]:
apple_prices.iloc[ :22]

In [ ]:
### APPLE stock price
### past 12 years

aapl =\
(
    yf
    .download("AAPL",
              start = "2013-05-07",
              end = "2025-05-06")
    .droplevel("Ticker",
               axis = 1)
    [["Close"]]
)

In [ ]:
aapl

In [ ]:
### Noise-reduced MAs 

### Shorter-term --> 22 days
### Longer-term --> 250 days

aapl["ma_22"] =\
(
    aapl
    ["Close"]
    .rolling(window = 22)
    .mean()
)

aapl["ma_250"] =\
(
    aapl
    ["Close"]
    .rolling(window = 250)
    .mean()
)

In [ ]:
aapl

In [ ]:
%whos

In [ ]:
### Write your code like Pythonista

permutation_of_windows = [12, 22, 48, 56, 120, 220, 250] # hyper-parameter to tune for optimization

In [ ]:
aapl.head(3)

In [ ]:
for i in permutation_of_windows:
    aapl[f"sma_{i}"] =\
    (
        aapl
        ["Close"]
        .rolling(window = i)
        .mean()
    )

In [ ]:
aapl

> Here, the SMAs can be used to generate positions to implement a trading strategy.

> Building a trading strategy will be discussed below. But here, let’s visualize a long position by a value of 1 and a short position by a value of –1.

> The change in the position can be visually detected, when the two lines representing the SMA time series cross.

## <a id = "p2">2.</a>  <font color = "green"> Simple Momentum Trading Strategy </font>  [back to table of contents](#top)

<a id="trading"></a>
### A Gentle Introduction to Stock Trading Strategies: A Simple Momentum Trading Strategy

> Now, we will discuss the development of a simple `momentum` strategy. We’ll go through that development step by step, firstly by formulating and programming a simple algorithmic trading strategy. We’ll then backtest our strategy, assessing its performance with the `pandas` library.

> Building a trading strategy requires multiple stages of work. As you will see later in our course, when you learn about machine learning (no pun intended here), building a trading strategy is an iterative process of training, tuning, optimizing, and assessing the performance of your model.

> As a starting point in learning about stock trading strategies, you will learn momentum strategy (a.k.a. divergence or trend trading). This strategy is based on a theory that movement of a quantity will continue in its current direction. Thus, stocks have momentum, or upward or downward trends, and you can detect and exploit those trends.

#### A moving average crossover and a dual moving average crossover are examples of the momentum strategy.

- The `moving average crossover` is when the price of an asset moves from one side of a moving average to the other. This `crossover` denotes a `change in momentum`, and can be used as the point at which to make the decision to `enter or exit` the market.

<br>

- A `dual moving average crossover` occurs when a short-term average crosses a long-term average. This signal is taken as identifying that momentum is shifting in the direction of the short-term average. A `buy signal` is when the short-term average crosses the long-term average and scores above it. A `sell signal` is presented by a short-term average crossing a long-term average and falling below it.

> Here, when you go long, you think that the stock price will go up and will sell at a higher price in the future (buy signal)

> When you go short, you sell your stock, expecting to be able to buy it back at a lower price and realize a profit (sell signal).

> Below, we will go through the strategy building step by step.

In [ ]:
%whos

#### Step 1: `Define` two lookback periods, of `short` and `long` duration

> Here we will create two variables and assign one integer per variable.

> It is obvious, but make sure that the integer you assign to the short window is shorter than the integer you assign to the long window variable 🙂

In [ ]:
short = 22 # this is a NOT fixated value
long = 84

In [ ]:
short_long_periods = [10, 22, 64, 84, 120, 250]

In [ ]:
import yfinance as yf

In [ ]:
yf.__version__

In [ ]:
nvda =\
(
    yf
    .download(["NVDA"],
              start = dt.datetime(2014, 5, 8),
              end = dt.datetime(2024, 5, 7)
             )
    ["Close"]
)

nvda

#### Step 2: Create an empty signals DataFrame

> Make sure to copy the index of your stock data so that we can start calculating the daily `buy or sell` signal 👀 

In [ ]:
BUY_or_SELL =\
(
    pd
    .DataFrame(index = nvda.index) # investment universe
)

In [ ]:
BUY_or_SELL

> You might want to make a column named `BUY_or_SELL` in your empty DataFrame, and set the value for all rows in that column to 0.0.

In [ ]:
BUY_or_SELL["BUY_or_SELL"] = 0.0
BUY_or_SELL

> Steps 1 and 2 above set the stage for our main work.

#### Step 3: Assign the set of short and long SMAs (over the respective short and long time windows)

> Here, please use the function `rolling()` to initiate your rolling window calculations. 

> Specifically, you will set the following three arguments.

* `window` 
* `min_period`
* `center`  

> Input either SHORT or LONG, with `1` as the minimum number of data points in the window that are required to have a value, and False, so that the labels are not set at the center of the window.

> Here, don’t forget to chain the function `mean()` so that we can calculate the rolling mean.

```python
moving_averages["ma_"] =\
(
    DF
    ["instrument"]
    .rolling(window = int # ,
            # min_periods = int,
            # center = bool
            )
    .mean()
)
```

In [ ]:
nvda.columns

In [ ]:
help(nvda.rename)

In [ ]:
nvda =\
(
    nvda              # "old": "new"  
    .rename(columns = {"NVDA": "price"}
           )
)

In [ ]:
nvda

In [ ]:
%whos

> Now you have calculated the mean average of the short and long windows.

In [ ]:
short

In [ ]:
### Shorter-term simple moving averages

BUY_or_SELL["shorter_SMA"] =\
(                 # EWMA
    nvda
    ["price"]
    .rolling(window = short,
             min_periods = 1, # there will be no missing values for moving averages
             center = False)
    .mean()
)

In [ ]:
BUY_or_SELL.head(3)

In [ ]:
%who

In [ ]:
long

In [ ]:
short_long_periods

In [ ]:
# BUY_or_SELL["SMA_120"] =\
# (
#     nvda
#     ["price"]
#     .rolling(window = 120)
#     .mean()
# )

In [ ]:
# BUY_or_SELL["SMA_250"] =\
# (
#     nvda
#     ["price"]
#     .rolling(window = 250)
#     .mean()
# )

In [ ]:
# Pythonian-way of assigning various sma calculations

for i in short_long_periods:
    BUY_or_SELL[f"SMA_{i}"] =\
    (
        nvda
        ["price"]
        .rolling(window = i,
                 min_periods = 1)
        .mean()
    )

In [ ]:
BUY_or_SELL

In [ ]:
## Longer-term

BUY_or_SELL["longer_SMA"] =\
(
    nvda
    ["price"]
    .rolling(window = long,
             min_periods = 1,
             center = False)
    .mean()
)

BUY_or_SELL.iloc[ :22]

#### Step 4: Create a signal when the short moving average crosses the long moving average

> Note that this is only for the period greater than the shortest moving average window. [Recall that this is the step we performed above during our refresher on rolling statistics.](#rolling)

In [ ]:
long

```python
DF["COLUMN"][how_many_values_from_the_beginning:-how_many_values_from_the_end] # slicing off
```

In [ ]:
BUY_or_SELL["BUY_or_SELL"][long: ] =\
( 
    np # if else        # RULE       , True, False
    .where(BUY_or_SELL["shorter_SMA"][long: ] > BUY_or_SELL["longer_SMA"][long: ],
           1.0, 0.0 # team, we are not short-selling when momentum is lost --> pay attention 
           # 1.0 being momentum is activated
           # 0.0 being momentum is lost
           )
)

In [ ]:
BUY_or_SELL

#### Step 5: Calculate the difference between the signals to make trading orders

> We can differentiate between long and short positions, within the column of our DF, where we are `buying` or `selling` stock.

In [ ]:
toy_DF =\
(
    pd
    .DataFrame(
        {
            "signal": [0,0,0,1,0,1,1,1,0,1,0]
        }
    )
)

In [ ]:
toy_DF

In [ ]:
toy_DF["position"] =\
(
    toy_DF
    .diff()
)

In [ ]:
toy_DF

In [ ]:
BUY_or_SELL.columns

In [ ]:
BUY_or_SELL["positions"] =\
(
    BUY_or_SELL["BUY_or_SELL"]
    .diff()
)

BUY_or_SELL[['BUY_or_SELL', 'shorter_SMA', 'longer_SMA', 'positions']]

> Print the signals DataFrame and inspect the results. `Make sure to fully digest what the positions and the signal columns denote in this DF.`

> When you have taken the time to understand the results of the trading strategy, visualize the short and long moving averages, along with the buy and sell signals.

In [ ]:
f =\
(
    plt
    .figure(figsize = [16,8]
           )
)

sub=\
(
    f
    .add_subplot(111,
                 ylabel = "Stock Price")
)

# Price Action (Daily)

(
    nvda
    ["price"]
    .plot(ax = sub,
          lw = 0.70,
          color = "grey")
)

# Moving Averages(Shorter vs. Longer)

(
    BUY_or_SELL
    [["shorter_SMA", "longer_SMA"]]
    .plot(ax = sub,
          lw = 0.80,
          style = ["--", "--"]
         )
)

# BUY Signal

(
    sub
    .plot(BUY_or_SELL[BUY_or_SELL.positions == 1.0].index,
          BUY_or_SELL.shorter_SMA[BUY_or_SELL.positions == 1.0],
          "^",
          color = "green",
          markersize = 12)
)

# SELL Signal

(
    sub
    .plot(BUY_or_SELL[BUY_or_SELL.positions == -1.0].index,
          BUY_or_SELL.shorter_SMA[BUY_or_SELL.positions == -1.0],
          "v",
          color = "red",
          markersize = 12)
)

## <a id = "p3">3.</a>  <font color = "green"> Backtesting Trading Strategy </font>  [back to table of contents](#top)

> Now let’s backtest our trading strategy and assess its performance.

#### Elements of Backtesting

- `Data interface` to access your data (here, pandas or pandas-datareader).
- `Strategy` to formulate a signal to go long or short.
- `Portfolio` to generate orders and manage profit and loss (`PnL`).
- `Execution interface` to send orders to the broker and get confirmation that the stock was bought or sold.

> Here, as a starting point in your learning, we will consider the first three items above (but not the execution interface, yet).

> Let’s learn how to create a portfolio which can generate orders and manages the PnL, with yet another `step by step` approach (Yay~) 

#### Step 1: Create a variable (`our_capital`) to set our initial capital, along with a new DataFrame, `our_position`

> We copy the index from another DataFrame.

> This is the signals DataFrame, where we want to consider the period for which we have generated signals.

In [ ]:
our_capital = 7e6
our_capital

In [ ]:
BUY_or_SELL

In [ ]:
our_position =\
(
    pd
    .DataFrame(index = BUY_or_SELL.index)
)

our_position

#### Step 2: Create a new column, NVDA, in `our_position`

> On the days that the `BUY_or_SELL` signal is 1 and the short moving average crosses the long moving average (for the period greater than the shortest moving average window), we’ll buy 200 shares.

> On the days on which the signal is zero, the final result will be zero.

In [ ]:
our_position["nvda"] =\
(
    BUY_or_SELL["BUY_or_SELL"]
    * 200
)

our_position

#### Step 3: Create a new DF portfolio to store the market value of an open position

In [ ]:
toy_DF_for_multiply_method =\
(
    pd
    .DataFrame(
        {"A": [10, 50, 70],
         "B": [1, 5, 7]
        }
    )
)

In [ ]:
toy_DF_for_multiply_method

In [ ]:
sample_nvda =\
(
    nvda
    .reset_index()
    .iloc[ :3]
    [["price"]]
)

In [ ]:
sample_nvda

In [ ]:
toy_DF_for_multiply_method

In [ ]:
sample_nvda

In [ ]:
(
    toy_DF_for_multiply_method
    .multiply(sample_nvda["price"],
              axis = 0)
)

In [ ]:
portfolio =\
(
    our_position
    .multiply(nvda["price"],
              axis = 0)
)

portfolio

#### Step 4: Create a DF that stores the `difference_in_shares_owned`

In [ ]:
difference_in_shares_owned =\
(
    our_position.diff()
)

In [ ]:
difference_in_shares_owned

#### Step 5: Create a new column, `our_holdings`

> This column will store the value of the positions or shares we have bought, multiplied by the `Adj Close` price.

In [ ]:
portfolio["our_holdings"] =\
(
    our_position
    .multiply(nvda["price"],
              axis = 0)
    .sum(axis = 1)
)

portfolio

#### Step 6: Create a new column, `our_cash`

> This is the capital remaining to spend.

> It can be calculated by taking `our_capital` and subtracting `our_holdings` (the price that we paid when buying stock).

In [ ]:
difference_in_shares_owned

In [ ]:
portfolio["our_cash"] =\
(
    our_capital - (difference_in_shares_owned
                   .multiply(nvda["price"],
                             axis = 0)
                   .sum(axis = 1)
                  ).cumsum()
)

In [ ]:
portfolio

#### Step 7: Create a `total` column for `our_portfolio` DF

> This will contain the `sum` of `our_cash` and `holdings that we own`.

In [ ]:
portfolio["total"] = portfolio["our_cash"] + portfolio["our_holdings"]

portfolio

#### Step 8: Create a `returns` column for our_portfolio DF

$$
    r_t = \frac{(P_t - P_{t-1})}{P_{t-1}}
$$

In [ ]:
nvda_price_t0 =\
( # the first day price in our investment universe
    nvda
    [["price"]]
    .iloc[0]
)

In [ ]:
nvda_price_t1 =\
( # the second day price in our investment universe
    nvda
    [["price"]]
    .iloc[1]
)

In [ ]:
simple_daily_return_on_the_second_day =\
(
    nvda_price_t1 
    /
    nvda_price_t0
    - 1
)

In [ ]:
simple_daily_return_on_the_second_day

In [ ]:
portfolio["return"] =\
(
    portfolio
    ["total"] # denotes the total value of our portfolio
    .pct_change() # simple daily returns
)

In [ ]:
portfolio

#### VISUALIZE

In [ ]:
f=\
(
    plt
    .figure(figsize = [16, 9]
           )
)

ax =\
(
    f
    .add_subplot(111,
                 ylabel = "Value of Our Porfolio (USD; unit = Million)")
)

# Total Value of Our Portfolio

(
    portfolio
    ["total"]
    .plot(ax = ax,
          color = "grey",
          lw = 0.80)
)

# Signal (BUY)

(
    ax
    .plot(portfolio.loc[BUY_or_SELL.positions == 1.0].index,
          portfolio.total[BUY_or_SELL.positions == 1.0],
          "^",
          color = "green",
          markersize = 12)
)

# Signal (SELL)

(
    ax
    .plot(portfolio.loc[BUY_or_SELL.positions == -1.0].index,
          portfolio.total[BUY_or_SELL.positions == -1.0],
          "v",
          color = "red",
          markersize = 12)
)

> Now we have learned a trading strategy and backtested it. 

> Note that this is not the end of your trading strategy learning. You might want to improve your strategy. 

> Later you will learn how to improve the model, using machine learning algorithms such as KMeans, k-Nearest Neighbors (KNN), and Classification or Regression Trees.

> You might want to improve your trading strategy by working with multi-symbol portfolios. As we’ll discuss when assessing our moving average crossover strategy, incorporating only one company or symbol into your strategy rarely provides much information.

> You should use a risk management framework or event-driven backtesting to help mitigate the foresight (lookahead) bias.

## <a id = "p4">4.</a>  <font color = "green"> Performance Metrics </font>  [back to table of contents](#top)

### Performance Metrics: Assessing Our Moving Average Crossover Strategy

> Let’s quickly assess our simple trading strategy by utilizing what pandas offers:

- `Sharpe ratio`;
- `maximum drawdown`;
- `compound annual growth rate (CAGR)`

> Let’s start with the Sharpe ratio. Here, we won’t take the risk-free rate in estimating it.

> Please extend the original trading strategy with more data (from other companies), when using Sharpe ratio.

### Mathematical Definition of Sharpe Ratio

$$
    \begin{equation}
    \text{Sharpe Ratio} = \frac{\text{Expected Portfolio Return} - \text{Risk-Free Rate}}
    {\text{Porfolio Standard Deviation}}
    \end{equation}
$$

In [ ]:
maang = ["META", "AAPL", "AMZN", "NFLX", "GOOG"]

In [ ]:
maang_prices =\
(
    yf
    .download(maang,
              start = dt.datetime(2014, 5, 9),
              end = dt.datetime(2025, 5, 8)
             )
    ["Close"]
)

In [ ]:
maang_prices

### Daily Return of MAANG Stocks

In [ ]:
daily_returns_MAANG =\
(
    maang_prices
    .pct_change()
)

In [ ]:
daily_returns_MAANG = daily_returns_MAANG.fillna(0)

In [ ]:
from lets_plot import *
LetsPlot.setup_html()

### One way to gauge volatility of returns: boxplot

In [ ]:
daily_returns_MAANG.plot(kind = "box")
plt.show()

## Calculating Mathematical Defition of Volatiliy with Python

In [ ]:
non_annualized_volatility =\
(
    daily_returns_MAANG
    .rolling(252)
    .std()    
)

In [ ]:
non_annualized_volatility

In [ ]:
annualized_volatility =\
(
    daily_returns_MAANG
    .rolling(252)
    .std()    
    *
    np.sqrt(252) # ANNUALIZED
)

In [ ]:
annualized_volatility

### Fat-tailed Nature of Stock as an asset class

In [ ]:
import scipy.stats as stats

In [ ]:
f = plt.figure(figsize = [5,5]
              )

ax = f.add_subplot(111)

(
    stats
    .probplot(daily_returns_MAANG["AAPL"],
              plot = ax,
              dist = "norm")
)

plt.show()

> Sharpe ratio is often compared to other stocks. Please extend the original trading strategy with more data (from other companies).

$$
    \begin{equation}
    \text{Sharpe Ratio} = \frac{\text{Average Daily Return}}{\text{Daily Standard Deviation}} \times
    \sqrt{252}
    \end{equation}
$$

In [ ]:
portfolio.columns

In [ ]:
Sharpe =\
(
    (portfolio["return"].mean() / portfolio["return"].std()
    ) * np.sqrt(252)
)

In [ ]:
Sharpe

### Sharpen Our Financial Decision Making?

#### <mark>Scenario: Investment returning 7% per year for 25 years.</mark>

In [ ]:
!pip install numpy-financial

In [ ]:
import numpy_financial as npf

### Step 1: Let's estimate future value of your investment
### <mark>NOTE that this does NOT mean the TRUE final value of your investment.</mark>

In [ ]:
starting_capital = 5e4

In [ ]:
starting_capital

In [ ]:
investment_future_value =\
(
    npf # fv() returns future value of your investment
    .fv(rate = 0.07,
        nper = 25,
        pmt = 0,
        pv = -starting_capital)
)

In [ ]:
print("Our investment will return of a total $"
      + 
      str(round(investment_future_value, 1)
         )
      + 
      " in 25 years.")

### <mark>TEAM, we SHOULD take into account the deprecation of monetary value!</mark>

### Step 2: After adjusting for inflation of 3.5% per year for 25 years, what's the worth of our invesment.

In [ ]:
discounted_value =\
(
    npf
    .pv(rate = 0.035, # inflation rate (average inflation rate over time) --> hyperbolic function
        nper = 25,
        pmt = 0,
        fv = investment_future_value
        )
)

In [ ]:
print("After adjusting for inflation, our invesment in worth $"
      + 
     str(round(-discounted_value, 1)
        )
      + 
      " in today's dollars."
     )

### Maximum Drawdown (MDD)

> Maximum drawdown is for measuring the largest single drop, from peak to bottom, in the value of a portfolio ahead of a new peak being achieved. The metric denotes the risk of a portfolio chosen according to a certain strategy.

In [ ]:
window = 252

In [ ]:
rolling_max =\
(
    nvda
    ["price"]
    .rolling(window = window,
             min_periods = 1)
    # .mean()
    .max()
)

In [ ]:
daily_drawdown =\
(
    (nvda["price"] 
     / 
     rolling_max)
    - 1.0
)

# We are now calculating daily drawdown
# ==> how much closing price of each day has dropped from the rolling maximum price

* If the `value = 0`, the current daily closing price is the rolling maximum
* If the `value < 0`, it denotes the current daily closing price is below the rolling maximum

In [ ]:
daily_drawdown # when is the maximum draw-down?

In [ ]:
max_daily_drawdown =\
(
    daily_drawdown # this is negative
    .rolling(window = window,
             min_periods = 1)
    .min() # lowest value
)

In [ ]:
fig = plt.figure(figsize = [16, 7]
                )

daily_drawdown.plot(color = "grey",
                    lw = 0.80)

max_daily_drawdown.plot(color = "red",
                        lw = 0.80)

plt.show()

### Compound Annual Growth Rate (CAGR)

> CAGR gives us a constant rate of return over a time period. 

The rate tells you what you really have at the end of your investment period.

- First, let’s calculate CAGR by dividing the investment’s ending value (EV) by its beginning value (BV).
- Then, raise the result to the power of 1/n, where n is the number of periods.
- Subtract 1 from the result. This is the CAGR.

$$ CAGR = (EV/BV)^{1/n} - 1 $$

$$
    \text{CAGR} = \left( \frac{\text{End Value}}{\text{Start}} \right)^{\frac{365}
    {\text{Total Days}}
    } - 1
$$

In [ ]:
days =\
(
    (nvda.index[-1] 
     -
     nvda.index[0]
    ).days
)

days

In [ ]:
CAGR =\
(
    (nvda["price"][-1] 
     / 
     nvda["price"][0]
    ) ** (365.0 / days)   
) - 1

In [ ]:
CAGR

#### Cautionary Tales

> Beyond testing a trading strategy, backtesting tests it on relevant historical data to see if it is a viable strategy before you make any moves. With backtesting, you can simulate and analyze the risk and profitability of trading with a specific strategy over a period of time.

But keep in mind that backtesting has some limitations.

- There could be external events, such as market regime shifts, which are regulatory changes or macroeconomic events, which will affect your backtesting.
- Liquidity constraints, such as a ban on short sales, could also influence your backtesting.
- You could overfit a model (optimization bias) when you ignore strategy rules because you think it’s better like that (interference), or you could accidentally introduce information into past data (foresight bias).

## <a id = "p5">5.</a>  <font color = "green"> What We Learned... </font>  [back to table of contents](#top)

> Team, the current lecture notes will serve as valuable reference materials in the field—your playbook and field manual. As you learn, please make active use of Markdown cells to document your insights and understanding. This practice will not only help you fully absorb the material but also increase its applicability in real-world contexts. 🙂

    - Learning Pointer 1.

    - Learning Pointer 2.

    - Learning Pointer 3.

> `Thank you for working with the script, Team 👍`